In [0]:
%run ../utils/upload_data

In [0]:
files = ["diabetes.csv"]
base_volume = "/Volumes/mlpractice/source/mlmodel/diabetes_data"
url = "https://raw.githubusercontent.com/kuljotSB/DatabricksUdemyCourse/refs/heads/main/MachineLearningModel"

In [0]:
# upload the dataset to volume
upload_to_volume(url, files, base_volume)

In [0]:
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{base_volume}/diabetes.csv")
display(df)


In [0]:
# Split train-test data 
train_df, test_df = df.randomSplit([0.7, 0.3], seed=42)

print(f"Training dataset count: {train_df.count()}")
print(f"Test dataset count: {test_df.count()}")

In [0]:
# Perform Feature Engineering and data cleaning

# 1. Normalize the features
from pyspark.ml.feature import VectorAssembler, MinMaxScaler

numericFeatures = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]
assembeldVector = VectorAssembler(inputCols=numericFeatures, outputCol= 'numericFeatures')
vectorizedData = assembeldVector.transform(train_df)

minMax = MinMaxScaler(inputCol=assembeldVector.getOutputCol(), outputCol='normalizedFeatures')
scaledData = minMax.fit(vectorizedData).transform(vectorizedData)

# Compare the numeric values of the orginal and normalized dataset
compareData = scaledData.select("numericFeatures", 'normalizedFeatures')
display(compareData)



In [0]:
# Preparing the features and the labels 
from pyspark.sql import functions as F
prepedData = scaledData[F.col('normalizedFeatures').alias('features'), F.col('Outcome').alias('label')]

display(prepedData)

In [0]:
# Now we train the ML model using the training dataset
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=10, regParam=0.3)
lrModel = lr.fit(prepedData)
# Now print the model
print(lrModel)

In [0]:
# Test the model on the test dataset

# Prepare the test dataset: Normalize the features

vectorizedTestData = assembeldVector.transform(test_df)
scaledTestData = minMax.fit(vectorizedTestData).transform(vectorizedTestData)
prepedTestData = scaledTestData[F.col('normalizedFeatures').alias('features'), F.col('Outcome').alias('label')]

# Now predict the outcome
predictions = lrModel.transform(prepedTestData)
predicted = predictions.select('features', 'probability', F.col('prediction').astype('Int'), F.col('label').alias('actual'))
display(predicted)

In [0]:
# Let us Evaluate the model 
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
eval = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction')

# Simple Accuracy 
accuracy = eval.evaluate(predictions, {eval.metricName: 'accuracy'})
print(f"Accuracy: {accuracy}")

# Individual class merics 
labels = [0, 1]
print('\nIndividual class metrics: ')
for label in labels:
  print("Class %s" % (label))

  # precision
  precision = eval.evaluate(predictions, {eval.metricLabel: label, eval.metricName: 'precisionByLabel'})

  print('\tPrecision:', precision)

  # Recall
  recall = eval.evaluate(predictions, {eval.metricLabel: label, eval.metricName: 'recallByLabel'})

  print('\tRecall:', recall)

  # F1 score
  f1 = eval.evaluate(predictions, {eval.metricLabel: label, eval.metricName: 'fMeasureByLabel'})
  
  print('\tF1 Score:', f1)

# Weighted (overall metics)
overallPrecision = eval.evaluate(predictions, {eval.metricName: 'weightedPrecision'})
print('\nOverall metrics:', overallPrecision)

overallRecall = eval.evaluate(predictions, {eval.metricName: 'weightedRecall'})
print('\tRecall:', overallRecall)

overallF1 = eval.evaluate(predictions, {eval.metricName: 'weightedFMeasure'})
print('\tF1 Score:', overallF1)

In [0]:
# Let us Use Pipeline to infere prediction
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, MinMaxScaler 
from pyspark.ml.classification import LogisticRegression

# Get numeric cols
numericFeatures = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]

# Define the feature engineering and model training algorithm steps 
numericVector = VectorAssembler(inputCols=numericFeatures, outputCol='numericFeatures')
numericScaler = MinMaxScaler(inputCol=numericVector.getOutputCol(), outputCol='normalizedFeatures')
featureVector = VectorAssembler(inputCols=['normalizedFeatures'], outputCol= 'features')
lrAlgo = LogisticRegression(labelCol='Outcome', featuresCol='features', maxIter=10, regParam=0.3)

# Chain The steps as stages in a pipline
pipline = Pipeline(stages=[numericVector, numericScaler, featureVector, lrAlgo])

# Use the pipeline to prepare data and fit the model algorithm 
model = pipline.fit(train_df)
print('Model trained Successfully!')


In [0]:
# use the pipeline to inference prediction 
prediction = model.transform(test_df)
predicted = prediction.select("features", 'probability', 'prediction', F.col('Outcome').alias('actual'))
display(predicted)

In [0]:
# save the model in the volume
model.save(f"{base_volume}/diabetes_model/diabestsLR.model")

In [0]:
# now let us infere the saved model 
from pyspark.ml.pipeline import PipelineModel

persistedModel = PipelineModel.load(f"{base_volume}/diabetes_model/diabestsLR.model")

newData = spark.createDataFrame([
    {
        "Pregnancies": 8,
        "Glucose": 85,
        "BloodPressure": 65,
        "SkinThickness": 29,
        "Insulin": 0,
        "BMI": 26.6,
        "DiabetesPedigreeFunction": 0.672,
        "Age": 34
    }
])


prediction = persistedModel.transform(newData)
display(
    prediction.select("features", "probability", F.col("prediction").alias('predictedOutcome') )
)